# FFT and power spectral density (script)

Script version of the notebook on the documentation website, without the theory and schematics. Set `XFOIL_BIN` in the parameters cell to the path of your XFOIL binary, then run all cells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## The test signal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

# Known ingredients
A1, f1 = 1.0, 5.0      # amplitude, frequency [Hz]
A2, f2 = 0.5, 12.0
SIGMA_N = 1.0          # standard deviation of the noise

# Sampling
FS = 100.0             # sampling frequency [Hz]
T  = 60.0              # record length [s]

N = int(FS*T)
t = np.arange(N)/FS
rng = np.random.default_rng(0)
x = A1*np.sin(2*np.pi*f1*t) + A2*np.sin(2*np.pi*f2*t) + SIGMA_N*rng.standard_normal(N)

fig, ax = plt.subplots(figsize=(9, 3))
show = t <= 2.0
ax.plot(t[show], x[show], color='0.3', lw=1, label='signal $x(t)$')
ax.plot(t[show], (A1*np.sin(2*np.pi*f1*t) + A2*np.sin(2*np.pi*f2*t))[show],
        color='tab:blue', lw=1.5, label='sine waves only')
ax.set_xlabel('$t$ [s]'); ax.set_ylabel('$x$')
ax.set_title('First 2 s of the signal')
ax.legend(loc='upper right', fontsize=8); ax.grid(alpha=0.3)
plt.show()

## Sampling

In [ ]:
print(f"N = {N},  T = {T:g} s,  f_s = {FS:g} Hz")
print(f"Nyquist frequency     f_Ny = {FS/2:g} Hz")
print(f"Frequency resolution  Δf   = {1/T:.4f} Hz")

## FFT and amplitude spectrum

In [ ]:
X = np.fft.rfft(x)
f = np.fft.rfftfreq(N, d=1/FS)
amp = 2*np.abs(X)/N

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(f, amp, color='tab:blue', lw=0.8)
ax.set_xlabel('$f$ [Hz]'); ax.set_ylabel(r'$\hat A$')
ax.set_title('Amplitude spectrum')
ax.set_xlim(0, FS/2); ax.grid(alpha=0.3)
for fk, Ak in ((f1, A1), (f2, A2)):
    k = np.argmin(np.abs(f - fk))
    ax.annotate(f'{f[k]:g} Hz, $\\hat A$ = {amp[k]:.2f}', xy=(f[k], amp[k]), xytext=(f[k] + 2, amp[k]),
                fontsize=9, va='center')
plt.show()

## Welch's method

In [ ]:
S_per = 2*np.abs(X)**2/(FS*N)
S_per[0] /= 2
S_per[-1] /= 2

SEG = 1000                                    # segment length L (10 s)
f_w, S_w = welch(x, fs=FS, window='hann', nperseg=SEG, noverlap=SEG//2)

print(f"Expected variance            : {A1**2/2 + A2**2/2 + SIGMA_N**2:.4f}   (A1²/2 + A2²/2 + σn²)")
print(f"Variance of the signal       : {np.var(x):.4f}")
print(f"Area under the periodogram   : {np.trapezoid(S_per, f):.4f}")
print(f"Area under the Welch PSD     : {np.trapezoid(S_w, f_w):.4f}")
print(f"Welch: {N//(SEG//2) - 1} segments,  Δf = {FS/SEG:g} Hz")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.loglog(f[1:], S_per[1:], color='0.7', lw=0.6, label='periodogram (one FFT)')
ax.loglog(f_w[1:], S_w[1:], color='tab:blue', lw=1.5, label=f'Welch ($L$ = {SEG}, Hann, 50 % overlap)')
ax.axhline(2*SIGMA_N**2/FS, color='k', ls=':', lw=1, label=r'white-noise level $2\sigma_n^2/f_s$')
ax.set_xlabel('$f$ [Hz]'); ax.set_ylabel('$S(f)$')
ax.set_title('Power spectral density')
ax.set_xlim(f[1], FS/2)
ax.legend(fontsize=8, loc='lower left'); ax.grid(alpha=0.3, which='both')
plt.show()

## Premultiplied spectrum and dominant frequency

In [ ]:
E = S_w/np.trapezoid(S_w, f_w)          # normalised to unit area (unit variance)
fE = f_w*E
k_pk = np.argmax(fE[1:]) + 1
f_pk = f_w[k_pk]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.loglog(f_w[1:], fE[1:], color='tab:blue', lw=1.5)
ax.axvline(f_pk, color='tab:red', ls='--', lw=1, label=f'dominant frequency = {f_pk:g} Hz')
ax.set_xlabel('$f$ [Hz]'); ax.set_ylabel(r'$f\,E(f)$')
ax.set_title('Premultiplied, normalised spectrum')
ax.set_xlim(f_w[1], FS/2)
ax.legend(fontsize=9, loc='lower right'); ax.grid(alpha=0.3, which='both')
plt.show()